# 三一重工（600031.SH）—— 每日交易数据获取与可视化分析

本 Notebook 演示如何：
1. 通过 **Tushare Pro** API 获取 A 股日线行情数据
2. 对数据进行清洗与探索
3. 使用 **Matplotlib** 绘制收盘价走势图、成交量图等

> 数据范围：过去一年（2025-07-02 ~ 2026-07-02）

## 1. 环境准备 — 导入库

## 2. 从 Tushare Pro 获取数据

> 需要先注册 [Tushare Pro](https://tushare.pro) 并获取 API Token。
> 本笔记本默认从本地 CSV 读取已保存的数据，避免重复请求 API。
> 如需实时拉取，请将下方 `USE_CACHED` 改为 `False` 并填入你的 Token。

In [ ]:
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti TC", "Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False
import os
import pandas as pd
# 配置参数
TS_CODE = '600031.SH'          # 三一重工
START_DATE = '20250702'        # 过去一年
END_DATE = '20260702'          # 今天
CSV_PATH = '/Users/xuyajing/Desktop/AI/Quant/sany_600031_daily.csv'  # 本地缓存文件（绝对路径）

# ========== 修改这里切换数据源 ==========
USE_CACHED = True   # True = 读本地 CSV，False = 从 Tushare API 拉取
# =====================================

if USE_CACHED:
    # 从本地 CSV 读取
    if os.path.exists(CSV_PATH):
        df = pd.read_csv(CSV_PATH)
        # 确保日期按时间升序排列
        df['trade_date'] = pd.to_datetime(df['trade_date'])
        df.sort_values('trade_date', inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f'✅ 从本地 CSV 读取 {len(df)} 条记录')
    else:
        print('⚠️ 本地 CSV 不存在，请将 USE_CACHED 设为 False 从 API 拉取')
else:
    # 从 Tushare API 拉取（需先 pip install tushare）
    import tushare as ts
    TOKEN = '你的 Tushare Token'  # 请替换
    pro = ts.pro_api(TOKEN)
    
    df_raw = pro.daily(ts_code=TS_CODE, start_date=START_DATE, end_date=END_DATE)
    df = df_raw.sort_values('trade_date').reset_index(drop=True)
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    print(f'✅ 从 Tushare API 获取 {len(df)} 条记录')

print('✅ 环境准备完成')
df.head()


## 3. 数据概览

In [1]:
print('--- 数据基本信息 ---')
print(f'行数: {len(df)}')
print(f'日期范围: {df["trade_date"].min().date()} ~ {df["trade_date"].max().date()}')
print(f'列名: {list(df.columns)}\n')

print('--- 字段类型 ---')
print(df.dtypes)
print()

print('--- 描述性统计 ---')
df[['open', 'high', 'low', 'close', 'vol', 'amount', 'pct_chg']].describe().round(2)

--- 数据基本信息 ---


NameError: name 'df' is not defined

In [2]:
first_close = df['close'].iloc[0]
last_close = df['close'].iloc[-1]
total_return = (last_close - first_close) / first_close * 100
max_close = df['close'].max()
min_close = df['close'].min()
avg_close = df['close'].mean()
max_date = df.loc[df['close'].idxmax(), 'trade_date'].date()
min_date = df.loc[df['close'].idxmin(), 'trade_date'].date()

print(f'📊 关键统计指标')
print(f'{"━"*40}')
print(f'起始收盘价  ({df["trade_date"].iloc[0].date()}): ¥{first_close:.2f}')
print(f'最新收盘价  ({df["trade_date"].iloc[-1].date()}): ¥{last_close:.2f}')
print(f'期间涨跌幅:    {total_return:+.2f}%')
print(f'最高收盘价:    ¥{max_close:.2f}（{max_date}）')
print(f'最低收盘价:    ¥{min_close:.2f}（{min_date}）')
print(f'均价:        ¥{avg_close:.2f}')
print(f'{"━"*40}')

NameError: name 'df' is not defined

## 4. 绘制收盘价曲线

> A 股惯例：**涨 = 红色**，**跌 = 绿色**

In [3]:
fig, ax = plt.subplots(figsize=(16, 6))

# 根据收盘价涨跌着色
colors = ['#e74c3c' if c >= o else '#27ae60' 
          for c, o in zip(df['close'], df['open'])]

ax.plot(df['trade_date'], df['close'], 
        color='#e74c3c', linewidth=1.5, alpha=0.85, label='收盘价')

# 填充渐变面积
ax.fill_between(df['trade_date'], df['close'], df['close'].min(),
                alpha=0.08, color='#e74c3c')

# 均价线
avg = df['close'].mean()
ax.axhline(y=avg, color='#5b8ff9', linestyle='--', linewidth=1, alpha=0.7)
ax.text(df['trade_date'].iloc[-1], avg, f'  均价 ¥{avg:.2f}', 
        va='bottom', fontsize=11, color='#5b8ff9')

# 标注最高/最低点
max_idx = df['close'].idxmax()
min_idx = df['close'].idxmin()
ax.annotate(f'最高 ¥{df["close"].max():.2f}',
            xy=(df['trade_date'].iloc[max_idx], df['close'].max()),
            xytext=(20, 20), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='#e74c3c'),
            fontsize=11, color='#e74c3c')
ax.annotate(f'最低 ¥{df["close"].min():.2f}',
            xy=(df['trade_date'].iloc[min_idx], df['close'].min()),
            xytext=(20, -25), textcoords='offset points',
            arrowprops=dict(arrowstyle='->', color='#27ae60'),
            fontsize=11, color='#27ae60')

# 格式化坐标轴
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.set_xlabel('交易日期', fontsize=12)
ax.set_ylabel('收盘价 (¥)', fontsize=12)
ax.set_title('三一重工 (600031.SH) 每日收盘价走势', fontsize=16, fontweight='bold', pad=15)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## 5. 成交量分析

In [4]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), 
                                gridspec_kw={'height_ratios': [3, 1]},
                                sharex=True)

# ---- 上：收盘价曲线 ----
ax1.plot(df['trade_date'], df['close'], color='#e74c3c', linewidth=1.5)
ax1.fill_between(df['trade_date'], df['close'], df['close'].min(),
                 alpha=0.08, color='#e74c3c')
ax1.axhline(y=avg, color='#5b8ff9', linestyle='--', linewidth=1, alpha=0.7)
ax1.set_ylabel('收盘价 (¥)', fontsize=12)
ax1.set_title('三一重工 (600031.SH) — 收盘价与成交量', fontsize=16, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.legend(['收盘价', f'均价 ¥{avg:.2f}'], loc='upper left')

# ---- 下：成交量柱状图 ----
colors_vol = ['#e74c3c' if df['pct_chg'].iloc[i] >= 0 else '#27ae60' 
              for i in range(len(df))]
ax2.bar(df['trade_date'], df['vol'] / 10000, color=colors_vol, alpha=0.7, width=0.8)
ax2.set_xlabel('交易日期', fontsize=12)
ax2.set_ylabel('成交量 (万股)', fontsize=12)
ax2.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## 6. 涨跌幅分布

In [5]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.hist(df['pct_chg'], bins=40, color='#5b8ff9', edgecolor='white', alpha=0.8)
ax.axvline(x=0, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_xlabel('涨跌幅 (%)', fontsize=12)
ax.set_ylabel('交易日数', fontsize=12)
ax.set_title('三一重工 — 每日涨跌幅分布', fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')

# 统计
up_days = (df['pct_chg'] > 0).sum()
down_days = (df['pct_chg'] < 0).sum()
flat_days = (df['pct_chg'] == 0).sum()
ax.text(0.98, 0.95, f'涨 {up_days} 天 | 跌 {down_days} 天 | 平 {flat_days} 天',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=12, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## 7. 移动平均线（MA10 & MA20）

In [6]:
df['MA10'] = df['close'].rolling(window=10).mean()
df['MA20'] = df['close'].rolling(window=20).mean()

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df['trade_date'], df['close'], color='#e74c3c', linewidth=1.5, alpha=0.7, label='收盘价')
ax.plot(df['trade_date'], df['MA10'], color='#f39c12', linewidth=1.5, label='MA10')
ax.plot(df['trade_date'], df['MA20'], color='#5b8ff9', linewidth=1.5, label='MA20')

ax.set_xlabel('交易日期', fontsize=12)
ax.set_ylabel('价格 (¥)', fontsize=12)
ax.set_title('三一重工 (600031.SH) — 移动平均线', fontsize=16, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3, linestyle='--')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

NameError: name 'df' is not defined

## 8. 总结

通过本 Notebook，我们完成了以下工作：

| 步骤 | 内容 |
|------|------|
| ✅ | 从 Tushare Pro 获取三一重工（600031.SH）过去一年的日线数据 |
| ✅ | 数据清洗与探索性分析 |
| ✅ | 绘制收盘价走势曲线（含高低点标注、均价线） |
| ✅ | 成交量分析（价格-成交量双面板） |
| ✅ | 涨跌幅分布直方图 |
| ✅ | 移动平均线（MA10 / MA20） |

---
*数据来源：[Tushare Pro](https://tushare.pro)*  |  *本分析仅供参考，不构成投资建议*